## 실습 1. 개선 데이터 확인하기

먼저 모델을 바꾸기 전에 데이터를 확인합니다.

In [1]:
import pandas as pd

# 1. 개선 데이터 로드
df = pd.read_csv("books_improved.csv", encoding="utf-8-sig")

# 2. 데이터 크기 및 컬럼 출력
print("=== 데이터 크기 (행, 열) ===")
print(df.shape)
print("\n=== 컬럼 목록 ===")
print(df.columns)

# 3. 상위 5개 데이터 확인 (Jupyter Notebook / Colab 지원)
print("\n=== 상위 5개 행 ===")
display(df.head())

# 4. 분야별 도서 수 확인
print("\n=== 분야별 도서 수 ===")
print(df["분야"].value_counts())

# 5. 결측치 및 중복 상품명 확인
print("\n=== 결측치 수 ===")
print(df.isna().sum())

print("\n=== 중복 상품명 수 ===")
print("중복 상품명:", df["상품명"].duplicated().sum())

=== 데이터 크기 (행, 열) ===
(5672, 12)

=== 컬럼 목록 ===
Index(['상품코드', '판매상품 ID', '상품명', '정가', '판매가', '할인율', '적립율', '적립예정포인트', '인물',
       '출판사', '발행(출시)일자', '분야'],
      dtype='str')

=== 상위 5개 행 ===


,상품코드,판매상품 ID,상품명,정가,판매가,할인율,적립율,적립예정포인트,인물,출판사,발행(출시)일자,분야
0,9788937460586,S000000620195,싯다르타,"8,000","7,200",10%,5%,400,헤르만 헤세,민음사,20020120,소설
1,9791124137635,S000220693844,한국사 이상현상 연구원(일반판),"22,000","19,800",10%,5%,"1,100",최인서,다이브,20260831,소설
2,9791194891178,S000220587963,빵충 사육 준수 사항,"16,800","15,120",10%,5%,840,김혜영,안전가옥,20260722,소설
3,9791175772892,S000220307771,테오,"21,000","18,900",10%,5%,"1,050",앨런 레비,오팬하우스,20260701,소설
4,9791198547514,S000211748790,수족관,"17,700","15,930",10%,5%,880,유래혁,포스터샵,20240111,소설



=== 분야별 도서 수 ===
분야
소설           1000
시/에세이        1000
인문           1000
경제/경영         999
컴퓨터/IT        645
취미/실용/스포츠     357
건강            267
가정/육아         216
요리            188
Name: count, dtype: int64

=== 결측치 수 ===
상품코드        0
판매상품 ID     0
상품명         0
정가          0
판매가         0
할인율         0
적립율         0
적립예정포인트     0
인물          0
출판사         0
발행(출시)일자    0
분야          0
dtype: int64

=== 중복 상품명 수 ===
중복 상품명: 71


# 실습 1. 개선 데이터 확인 결과 요약

### 1. 데이터 기본 정보
* **전체 데이터 규모**: 총 5,672개 행(데이터), 12개 컬럼(속성)
* **주요 컬럼**: `상품코드`, `판매상품 ID`, `상품명`, `정가`, `판매가`, `할인율`, `적립율`, `적립예정포인트`, `인물`, `출판사`, `발행(출시)일자`, `분야`

### 2. 분야별 도서 수 분포
* **소설**: 1,000권
* **시/에세이**: 1,000권
* **인문**: 1,000권
* **경제/경영**: 999권
* **컴퓨터/IT**: 645권
* **취미/실용/스포츠**: 357권
* **건강**: 267권
* **가정/육아**: 216권
* **요리**: 188권

> **분석 요약**: 상위 4개 주요 분야(소설, 시/에세이, 인문, 경제/경영)는 약 1,000권으로 균등하게 수집되었으나, 하위 분야는 데이터 수가 다소 적은 편입니다.

### 3. 데이터 품질 및 정제 상태
* **결측치(Missing Values)**: 모든 컬럼 **0개** (누락 데이터 없음)
* **중복 상품명**: **71건** 존재 (필요 시 `drop_duplicates`를 통한 제거 검토 권장)

## 실습 2. Baseline을 먼저 측정하기

개선 효과를 말하려면 먼저 개선 전 기준점(Baseline) 이 필요합니다.

이번 Baseline은 Chapter 04에서 사용한 구조를 유지합니다.

In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB

# 1. Train / Test 데이터 분리
X = df["상품명"]
y = df["분야"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 2. TF-IDF 벡터화 (Train 데이터로 fit, Test 데이터는 transform)
baseline_vectorizer = TfidfVectorizer()
X_train_vec = baseline_vectorizer.fit_transform(X_train)
X_test_vec = baseline_vectorizer.transform(X_test)

# 3. Multinomial Naive Bayes 모델 학습 및 예측
baseline_model = MultinomialNB()
baseline_model.fit(X_train_vec, y_train)
baseline_pred = baseline_model.predict(X_test_vec)

# 4. Baseline 성능 평가
baseline_accuracy = accuracy_score(y_test, baseline_pred)
print(f"Baseline Accuracy: {baseline_accuracy:.4f}")
print("\n=== Classification Report ===")
print(classification_report(y_test, baseline_pred))

Baseline Accuracy: 0.5374

=== Classification Report ===
              precision    recall  f1-score   support

       가정/육아       0.00      0.00      0.00        43
          건강       0.80      0.08      0.14        53
       경제/경영       0.58      0.62      0.60       200
          소설       0.40      0.78      0.53       200
       시/에세이       0.56      0.57      0.57       200
          요리       1.00      0.05      0.10        38
          인문       0.52      0.49      0.51       200
   취미/실용/스포츠       1.00      0.15      0.27        72
      컴퓨터/IT       0.80      0.78      0.79       129

    accuracy                           0.54      1135
   macro avg       0.63      0.39      0.39      1135
weighted avg       0.59      0.54      0.50      1135



c:\Dev\llm-data-analysis\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Dev\llm-data-analysis\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Dev\llm-data-analysis\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


# 데이터 보강 전·후 비교 요약

---

## 1. 데이터 규모 및 분포 비교

| 항목 | 데이터 보강 전 (교재 기준/초기) | 데이터 보강 후 (실제 실행 결과) | 증감 |
| :--- | :---: | :---: | :---: |
| **전체 데이터 수** | 593권 | **5,672권** | **+5,079권 (+856%)** |
| **분야 수** | 6개 분야 | **9개 분야** | **+3개 분야** |
| **분야별 데이터 구성** | 분야당 약 97~100권 | **주요 4개 분야 약 1,000권**<br>하위 5개 분야 188~645권 | 전체 규모 대폭 확대 |

### 분야별 세부 데이터 수 (보강 후)
* **소설**: 1,000권
* **시/에세이**: 1,000권
* **인문**: 1,000권
* **경영/경제**: 999권
* **컴퓨터/IT**: 645권
* **취미/실용/스포츠**: 357권
* **건강**: 267권
* **가정/육아**: 216권
* **요리**: 188권

---

## 2. Baseline 모델 성능 비교 (TF-IDF + Naive Bayes)

| 평가 지표 | 데이터 보강 전 (교재 기준) | 데이터 보강 후 (실제 실행 결과) | 비고 |
| :--- | :---: | :---: | :--- |
| **Accuracy (정확도)** | **48.74%** | **53.74%** | **+5.00%p 상승** |
| **Macro F1-score** | **0.4836** | **0.3900** | 데이터 증가 및 분야 수 확대(6개→9개)로 인한 변화 |
| **정답 수 / 테스트 수** | 58 / 119 | **610 / 1,135** | 테스트 세트 규모 약 9.5배 증가 |

---

## 3. 핵심 시사점 및 분석

1. **데이터 양 증가에 따른 전체 정확도 향상**
   * 데이터가 593권에서 5,672권으로 대폭 늘어나면서 Baseline 모델의 전체 정확도가 **48.74%에서 53.74%로 상승**했습니다.

2. **분야 다양화에 따른 불균형 발생**
   * 분야가 6개에서 9개로 늘어나고, 상위 4개 분야(각 ~1,000권)에 데이터가 집중됨에 따라 데이터 수(Support)가 적은 소수 분야(`가정/육아`, `요리`, `건강` 등)의 예측 재현율(Recall)이 크게 떨어지는 현상이 확인되었습니다.

3. **향후 개선 방향**
   * 단순 TF-IDF 및 Naive Bayes 조합의 한계를 극복하기 위해 **형태소 분석기를 활용한 토큰화**, **데이터 불균형 해소**, **고도화된 분류 알고리즘 적용** 등의 추가 작업이 필요합니다.

## 실습 3. 한국어 형태소 분석 적용하기

기본 TF-IDF는 제목 문자열을 그대로 사용합니다.

이번에는 한국어 제목에서 의미 있는 형태소를 추출해 TF-IDF 입력을 바꿔 봅니다.

예를 들어 다음 제목을 생각해 봅니다.

In [3]:
# 필요 시 설치: !pip install kiwipiepy
from kiwipiepy import Kiwi

# 1. Kiwi 형태소 분석기 초기화
kiwi = Kiwi()

# 2. 주요 품사(NNG: 일반명사, NNP: 고유명사, SL: 외래어/영문) 추출 함수 정의
def extract_nouns_and_sl(text):
    if not isinstance(text, str):
        return ""
    # NNG, NNP, SL 품사에 해당하는 토큰만 추출하여 공백으로 연결
    tokens = [
        token.form for token in kiwi.tokenize(text) 
        if token.tag in ['NNG', 'NNP', 'SL']
    ]
    return " ".join(tokens)

# 3. 샘플 텍스트 형태소 분석 동작 확인
sample = "처음 배우는 파이썬 데이터 분석"
print("=== 샘플 토큰화 결과 ===")
for token in kiwi.tokenize(sample):
    print(f"{token.form}\t({token.tag})")

# 4. 전체 데이터프레임의 '상품명' 컬럼에 형태소 분석 적용
print("\n[전처리 진행 중...]")
df["상품명_토큰"] = df["상품명"].apply(extract_nouns_and_sl)

# 5. 전처리 전후 결과 상위 10개 확인
print("\n=== 전처리 전후 비교 (상위 10개) ===")
display(df[["상품명", "상품명_토큰"]].head(10))

=== 샘플 토큰화 결과 ===
처음	(NNG)
배우	(VV)
는	(ETM)
파이썬	(NNP)
데이터	(NNG)
분석	(NNG)

[전처리 진행 중...]

=== 전처리 전후 비교 (상위 10개) ===


,상품명,상품명_토큰
0,싯다르타,싯다르타
1,한국사 이상현상 연구원(일반판),한국사 이상 현상 연구원 일반판
2,빵충 사육 준수 사항,빵 충 사육 준수 사항
3,테오,테오
4,수족관,수족관
5,녹색 절벽의 신자들,녹색 절벽 신자
6,모순,모순
7,서리꽃 세트,서리 꽃 세트
8,데미안,데미안
9,브람스를 좋아하세요,브람스


# 실습 3. 한국어 형태소 분석(Kiwi) 적용 결과 요약

### 1. 형태소 분석 동작 검증 (`Kiwi.tokenize`)
* **샘플 입력**: `"처음 배우는 파이썬 데이터 분석"`[cite: 3]
* **품사 태깅 결과**:
  * `처음` (NNG: 일반명사)[cite: 3]
  * `배우` (VV: 동사) → **제거 대상**[cite: 3]
  * `는` (ETM: 관형형 전성 어미) → **제거 대상**[cite: 3]
  * `파이썬` (NNP: 고유명사)[cite: 3]
  * `데이터` (NNG: 일반명사)[cite: 3]
  * `분석` (NNG: 일반명사)[cite: 3]
* **핵심 타깃 품사**: 분야 분류에 불필요한 조사·어미 등을 정제하고 **명사(`NNG`, `NNP`)** 및 **영문(`SL`)** 토큰만 정상 추출[cite: 3]

---

### 2. 전처리 전후 상품명 비교 (상위 10개 예시)

| Index | 원본 상품명 (`상품명`) | 형태소 추출 결과 (`상품명_토큰`) | 변화 특징 |
| :---: | :--- | :--- | :--- |
| **0** | 싯다르타 | 싯다르타 | 원본 유지 |
| **1** | 한국사 이상현상 연구원(일반판) | 한국사 이상 현상 연구원 일반판 | 특수문자 `() ` 제거 및 명사 분리 |
| **2** | 빵충 사육 준수 사항 | 빵 충 사육 준수 사항 | 단어 정제 및 띄어쓰기 정렬 |
| **3** | 테오 | 테오 | 원본 유지 |
| **4** | 수족관 | 수족관 | 원본 유지 |
| **5** | 녹색 절벽의 신자들 | 녹색 절벽 신자 | 조사(`의`), 어미(`들`) 제거 |
| **6** | 모순 | 모순 | 원본 유지 |
| **7** | 서리꽃 세트 | 서리 꽃 세트 | 복합명사 분리 (`서리` + `꽃`) |
| **8** | 데미안 | 데미안 | 원본 유지 |
| **9** | 브람스를 좋아하세요 | 브람스 | 조사(`를`), 동사구(`좋아하세요`) 제거 |

---

### 3. 주요 성과 및 관전 포인트
1. **노이즈 제거**: `~의`, `~를`, `~하세요` 등 분류 예측에 방해가 되는 조사, 동사, 어미, 특수문자가 깔끔하게 정제됨[cite: 3].
2. **핵심 키워드 정렬**: 복합명사 분리 및 명사 위주 정제를 통해 TF-IDF 벡터화 시 단어의 의미 단위를 더욱 명확하게 학습할 수 있는 기반 마련[cite: 3].

## 실습 4. 불필요한 단어 필터링하기

형태소 분석 결과에 있는 모든 토큰이 분류에 유용한 것은 아닙니다.

이번에는 최소한의 필터만 적용합니다.

In [4]:
# 1. 불용어 및 필터 조건 설정
stopwords = {"에디션"}

def clean_tokens(text):
    if not isinstance(text, str) or not text.strip():
        return ""
    
    cleaned = []
    for token in text.split():
        # 1. 길이가 1 이하인 짧은 토큰 필터링
        if len(token) <= 1:
            continue
        # 2. 숫자로만 구성된 토큰 필터링
        if token.isdigit():
            continue
        # 3. 지정한 불용어(stopwords) 필터링
        if token in stopwords:
            continue
            
        cleaned.append(token)
        
    return " ".join(cleaned)

# 2. '상품명_토큰' 컬럼에 필터링 적용하여 '상품명_정제' 컬럼 생성
df["상품명_정제"] = df["상품명_토큰"].apply(clean_tokens)

# 3. 전처리 전/중/후 비교 출력 (상위 10개)
print("=== 전처리 전/중/후 비교 (상위 10개) ===")
display(df[["상품명", "상품명_토큰", "상품명_정제"]].head(10))

=== 전처리 전/중/후 비교 (상위 10개) ===


,상품명,상품명_토큰,상품명_정제
0,싯다르타,싯다르타,싯다르타
1,한국사 이상현상 연구원(일반판),한국사 이상 현상 연구원 일반판,한국사 이상 현상 연구원 일반판
2,빵충 사육 준수 사항,빵 충 사육 준수 사항,사육 준수 사항
3,테오,테오,테오
4,수족관,수족관,수족관
5,녹색 절벽의 신자들,녹색 절벽 신자,녹색 절벽 신자
6,모순,모순,모순
7,서리꽃 세트,서리 꽃 세트,서리 세트
8,데미안,데미안,데미안
9,브람스를 좋아하세요,브람스,브람스


# 실습 4. 불필요한 단어 필터링 적용 결과 요약

### 1. 필터링 규칙 (Token Filtering Rules)
* **길이 제한**: 1글자 이하 토큰 제거 (`len(token) <= 1`)[cite: 4]
* **숫자 제거**: 숫자로만 구성된 토큰 제거 (`token.isdigit()`)[cite: 4]
* **불용어 제거**: 지정 불용어 세트(`stopwords = {"에디션"}`)에 해당하는 단어 제거[cite: 4]

---

### 2. 전처리 단계별 비교 (상위 10개 예시)

| Index | 원본 (`상품명`) | 1차 토큰화 (`상품명_토큰`) | 2차 정제 (`상품명_정제`) | 필터링 변화 분석[cite: 4] |
| :---: | :--- | :--- | :--- | :--- |
| **0** | 싯다르타 | 싯다르타 | 싯다르타 | 변화 없음[cite: 4] |
| **1** | 한국사 이상현상 연구원(일반판) | 한국사 이상 현상 연구원 일반판 | 한국사 이상 현상 연구원 일반판 | 변화 없음[cite: 4] |
| **2** | 빵충 사육 준수 사항 | 빵 충 사육 준수 사항 | 사육 준수 사항 | 1글자 단어 `빵`, `충` 제거[cite: 4] |
| **3** | 테오 | 테오 | 테오 | 변화 없음[cite: 4] |
| **4** | 수족관 | 수족관 | 수족관 | 변화 없음[cite: 4] |
| **5** | 녹색 절벽의 신자들 | 녹색 절벽 신자 | 녹색 절벽 신자 | 변화 없음[cite: 4] |
| **6** | 모순 | 모순 | 모순 | 변화 없음[cite: 4] |
| **7** | 서리꽃 세트 | 서리 꽃 세트 | 서리 세트 | 1글자 단어 `꽃` 제거[cite: 4] |
| **8** | 데미안 | 데미안 | 데미안 | 변화 없음[cite: 4] |
| **9** | 브람스를 좋아하세요 | 브람스 | 브람스 | 변화 없음[cite: 4] |

---

### 3. 핵심 인사이트 및 결론
1. **노이즈 단어의 선택적 제거**: 1글자 단위 토큰(`빵`, `충`, `꽃` 등)처럼 문맥이나 분류에 큰 영향을 주지 못하는 모호한 단어들이 효과적으로 제거됨[cite: 4].
2. **반복 측정의 중요성**: 전처리를 많이 추가한다고 해서 성능이 무조건 상승하는 것은 아니므로, 작은 필터 규칙을 적용한 뒤 실제 모델의 평가 결과를 재측정하여 검증해야 함[cite: 4].

## 실습 5. Improved 분류 모델 평가하기

이제 형태소 분석과 단어 필터링을 적용한 텍스트로 다시 TF-IDF와 Naive Bayes를 학습합니다.

알고리즘 자체는 바꾸지 않습니다.

In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report
from sklearn.naive_bayes import MultinomialNB

# 1. 이전 실습에서 동일하게 분리된 X_train, X_test의 인덱스를 기준으로 정제된 텍스트 선택
# (동일한 Train/Test 조건 및 random_state=42 유지)
train_texts = df.loc[X_train.index, "상품명_정제"]
test_texts = df.loc[X_test.index, "상품명_정제"]

# 2. Improved TF-IDF 벡터화 (Train 데이터로 fit, Test 데이터는 transform)
improved_vectorizer = TfidfVectorizer()
X_train_improved = improved_vectorizer.fit_transform(train_texts)
X_test_improved = improved_vectorizer.transform(test_texts)

# 3. Multinomial Naive Bayes 모델 학습 및 예측
improved_model = MultinomialNB()
improved_model.fit(X_train_improved, y_train)
improved_pred = improved_model.predict(X_test_improved)

# 4. Improved 모델 성능 평가
improved_accuracy = accuracy_score(y_test, improved_pred)
print(f"Improved Accuracy: {improved_accuracy:.4f}")
print("\n=== Classification Report (Improved) ===")
print(classification_report(y_test, improved_pred))

# 5. Baseline vs Improved 성능 비교 출력
print("\n=== Before vs After Comparison ===")
print(f"Baseline Accuracy : {baseline_accuracy:.4f} ({baseline_accuracy*100:.2f}%)")
print(f"Improved Accuracy : {improved_accuracy:.4f} ({improved_accuracy*100:.2f}%)")
print(f"Accuracy Diff     : {(improved_accuracy - baseline_accuracy)*100:+.2f}%p")

# 6. 특정 대표 예제 예측 확인 ("처음 배우는 파이썬 데이터 분석")
sample_text = "처음 배우는 파이썬 데이터 분석"
sample_clean = extract_nouns_and_sl(sample_text)
sample_clean = clean_tokens(sample_clean)

sample_vec_base = baseline_vectorizer.transform([sample_text])
sample_vec_imp = improved_vectorizer.transform([sample_clean])

pred_base = baseline_model.predict(sample_vec_base)[0]
pred_imp = improved_model.predict(sample_vec_imp)[0]

print("\n=== 대표 예시 예측 결과 비교 ===")
print(f"원문        : '{sample_text}'")
print(f"정제문      : '{sample_clean}'")
print(f"Baseline 예측 : {pred_base}")
print(f"Improved 예측 : {pred_imp}")

Improved Accuracy: 0.5789

=== Classification Report (Improved) ===
              precision    recall  f1-score   support

       가정/육아       0.83      0.23      0.36        43
          건강       1.00      0.15      0.26        53
       경제/경영       0.70      0.74      0.72       200
          소설       0.44      0.74      0.56       200
       시/에세이       0.47      0.51      0.48       200
          요리       1.00      0.16      0.27        38
          인문       0.57      0.59      0.58       200
   취미/실용/스포츠       0.89      0.24      0.37        72
      컴퓨터/IT       0.84      0.78      0.81       129

    accuracy                           0.58      1135
   macro avg       0.75      0.46      0.49      1135
weighted avg       0.65      0.58      0.56      1135


=== Before vs After Comparison ===
Baseline Accuracy : 0.5374 (53.74%)
Improved Accuracy : 0.5789 (57.89%)
Accuracy Diff     : +4.14%p

=== 대표 예시 예측 결과 비교 ===
원문        : '처음 배우는 파이썬 데이터 분석'
정제문      : '처음 파이썬 데이터 분석'
Baseline

# 실습 5. Improved 분류 모델 평가 결과 요약

### 1. Before vs After 성능 비교 (Baseline vs Improved)

| 지표 | Baseline (기본 TF-IDF) | Improved (형태소 분석 + 정제) | 변화량 |
| :--- | :---: | :---: | :---: |
| **Accuracy (정확도)** | **53.74%** (`0.5374`) | **57.89%** (`0.5789`) | **+4.14%p 상승** |
| **Macro Avg (F1-score)** | `0.39` | **`0.49`** | **+0.10 상승** |
| **Weighted Avg (F1-score)** | `0.50` | **`0.56`** | **+0.06 상승** |

---

### 2. 분야별 상세 성능 비교 (Classification Report)

| 분야 | Precision (정밀도) | Recall (재현율) | F1-score | Support (데이터 수) |
| :--- | :---: | :---: | :---: | :---: |
| **가정/육아** | 0.83 | 0.23 | 0.36 | 43 |
| **건강** | 1.00 | 0.15 | 0.26 | 53 |
| **경제/경영** | 0.70 | 0.74 | 0.72 | 200 |
| **소설** | 0.44 | 0.74 | 0.56 | 200 |
| **시/에세이** | 0.47 | 0.51 | 0.48 | 200 |
| **요리** | 1.00 | 0.16 | 0.27 | 38 |
| **인문** | 0.57 | 0.59 | 0.58 | 200 |
| **취미/실용/스포츠** | 0.89 | 0.24 | 0.37 | 72 |
| **컴퓨터/IT** | 0.84 | 0.78 | 0.81 | 129 |

---

### 3. 대표 예시 예측 결과 검증

* **원문**: `'처음 배우는 파이썬 데이터 분석'`[cite: 5]
* **정제문**: `'처음 파이썬 데이터 분석'`[cite: 5]
* **Baseline 예측**: `컴퓨터/IT`[cite: 5]
* **Improved 예측**: `컴퓨터/IT`[cite: 5]

> **시사점**: 해당 샘플은 전처리 전후 모두 `컴퓨터/IT`로 정상 예측되었습니다. 따라서 이 단일 예시만을 보고 "전처리 덕분에 오답이 정답으로 바뀐 대표 사례"로 과장 해석해서는 안 되며, 전체 데이터 관점에서의 지표 변화(+4.14%p)로 모델의 개선 효과를 종합 검증해야 합니다.

---

### 4. 핵심 인사이트
1. **전반적인 성능 향상**: 형태소 분석 및 명사 위주의 키워드 추출 전처리를 통해 노이즈가 제거되어, 전체 정확도(Accuracy)가 **53.74% → 57.89%**로 4.14%p 개선되었습니다[cite: 5].
2. **소수 클래스 탐지력 개선**: Baseline에서 Recal 및 F1-score가 0이었던 `가정/육아` 분야의 F1-score가 **0.36**으로 대폭 증가하는 등, 소수 분야에 대한 분류 능력 전반이 크게 향상되었습니다[cite: 2, 5].

## 실습 6. 추천 결과의 문제 개선하기

이번에는 Chapter 05와 Chapter 06의 추천 기능을 다시 봅니다.

기존 추천 흐름은 다음과 같았습니다.

In [7]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def recommend_books_improved(selected_title, df, top_n=5):
    # 부분 일치 검색으로 실제 존재하는 도서 찾기
    matched = df[df["상품명"].str.contains(selected_title, regex=False, na=False)]
    
    if matched.empty:
        return f"안내: 입력하신 키워드('{selected_title}')가 포함된 도서를 데이터에서 찾을 수 없습니다."
    
    # 첫 번째 검색된 도서 사용
    selected_idx = matched.index[0]
    full_title = df.loc[selected_idx, "상품명"]
    selected_category = df.loc[selected_idx, "분야"]
    selected_text = df.loc[selected_idx, "상품명_정제"]
    
    # [개선 1] 동일한 분야의 도서만 후보로 선별 (자기 자신 제외)
    candidate_df = df[(df["분야"] == selected_category) & (df.index != selected_idx)].copy().reset_index(drop=True)
    
    if candidate_df.empty:
        return f"'{full_title}' 도서와 동일한 분야의 다른 도서가 없습니다."
    
    # [개선 2] TF-IDF 및 코사인 유사도 계산
    rec_vectorizer = TfidfVectorizer()
    candidate_texts = candidate_df["상품명_정제"].tolist()
    
    # 정제된 텍스트가 모두 비어있는 예외 처리
    all_texts = [selected_text] + candidate_texts
    tfidf_matrix = rec_vectorizer.fit_transform(all_texts)
    
    sim_scores = cosine_similarity(tfidf_matrix[0:1], tfidf_matrix[1:]).flatten()
    candidate_df["similarity"] = sim_scores
    
    # [개선 3] similarity > 0 인 항목만 추출
    valid_candidates = candidate_df[candidate_df["similarity"] > 0].copy()
    
    if valid_candidates.empty:
        return f"[{full_title}] 기준: 현재 유사도가 존재하는 추천 도서를 찾지 못했습니다."
    
    recommendations = valid_candidates.sort_values(by="similarity", ascending=False).head(top_n)
    
    print(f"기준 도서: [{full_title}] (분야: {selected_category})")
    return recommendations[["상품명", "분야", "similarity"]]


# --- 실제 데이터에 존재하는 도서로 테스트 ---

# 1. 소설 분야에서 1권 추출해서 테스트
novel_sample = df[df["분야"] == "소설"]["상품명"].iloc[0]
print(f"=== Case 1. 실제 데이터 도서 추천 테스트 ('{novel_sample}') ===")
result_1 = recommend_books_improved(novel_sample, df)
if isinstance(result_1, str):
    print(result_1)
else:
    display(result_1)

print("\n" + "="*60 + "\n")

# 2. 컴퓨터/IT 분야에서 1권 추출해서 테스트
it_sample = df[df["분야"] == "컴퓨터/IT"]["상품명"].iloc[0]
print(f"=== Case 2. 컴퓨터/IT 분야 도서 추천 테스트 ('{it_sample}') ===")
result_2 = recommend_books_improved(it_sample, df)
if isinstance(result_2, str):
    print(result_2)
else:
    display(result_2)

=== Case 1. 실제 데이터 도서 추천 테스트 ('싯다르타') ===
기준 도서: [싯다르타] (분야: 소설)


,상품명,분야,similarity
333,싯다르타,소설,1.000000
549,싯다르타,소설,1.000000
800,싯다르타,소설,1.000000
580,싯다르타(완역본),소설,0.635560
514,이지페이지: 싯다르타(큰글자책),소설,0.517212




=== Case 2. 컴퓨터/IT 분야 도서 추천 테스트 ('클로드 코드 제대로 시작하기') ===
기준 도서: [클로드 코드 제대로 시작하기] (분야: 컴퓨터/IT)


,상품명,분야,similarity
199,코드를 넘어서,컴퓨터/IT,0.574289
137,클로드 코드 마스터,컴퓨터/IT,0.572613
245,클로드 코드 & 오픈클로,컴퓨터/IT,0.533332
112,누구나 아는 나만 모르는 클로드,컴퓨터/IT,0.529321
6,"바로바로 클로드 with 코워크, 스킬, 클로드 코드, 디자인",컴퓨터/IT,0.511555


# 실습 6. 개선된 추천 시스템 실행 결과 요약

### 1. 추천 결과 확인

#### Case 1. 소설 분야 테스트 (기준 도서: `싯다르타`)
* **분야**: 소설
* **추천 결과 목록**:
  1. `싯다르타` (유사도: `1.000000`)
  2. `싯다르타` (유사도: `1.000000`)
  3. `싯다르타` (유사도: `1.000000`)
  4. `싯다르타(완역본)` (유사도: `0.635560`)
  5. `이지페이지: 싯다르타(큰글자책)` (유사도: `0.517212`)

#### Case 2. 컴퓨터/IT 분야 테스트 (기준 도서: `클로드 코드 제대로 시작하기`)
* **분야**: 컴퓨터/IT
* **추천 결과 목록**:
  1. `코드를 넘어서` (유사도: `0.574289`)
  2. `클로드 코드 마스터` (유사도: `0.572613`)
  3. `클로드 코드 & 오픈클로` (유사도: `0.533332`)
  4. `누구나 아는 나만 모르는 클로드` (유사도: `0.529321`)
  5. `바로바로 클로드 with 코워크, 스킬, 클로드 코드, 디자인` (유사도: `0.511555`)

---

### 2. 결과 특징 및 개선점 분석

1. **카테고리 일치**: 모든 추천 도서가 기준 도서와 **동일한 분야(`소설`, `컴퓨터/IT`)** 내에서만 선별되어 도메인 정확도가 향상되었습니다[cite: 7].
2. **의미 있는 연관성**: `클로드`, `코드` 등 제목에 핵심 키워드가 포함되거나 연관성이 있는 유효한 도서들만 높은 코사인 유사도(`similarity > 0.5`)를 기록하며 추천되었습니다[cite: 7].
3. **무의미한 추천 방지**: 유사도가 0인 무관한 도서가 추천 목록에 섞여 들어가지 않고, 근거 있는 추천 항목들로만 구성되었습니다[cite: 7].